In [1]:
%load_ext autoreload
%autoreload 2
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
import warnings
warnings.filterwarnings("ignore")

import os
import pandas as pd
import sys
sys.path.insert(0, '/home/kat/Repos/SALSA/')

In [2]:
seedy = 666
df = pd.read_csv('data/chembl_valid_10to30atoms.csv')

samp_size = 1000

df_samp = df.sample(samp_size, random_state=seedy)
df_samp['len'] = df_samp.smiles.apply(lambda x: len(x))
df_samp = df_samp.sort_values(by='len',ignore_index=True)

In [3]:
def get_augset_from_src(k,v,aug_iter):
        
    src_idx = v[0]
    anc_idx = v[1]
    smi = v[4]

    augs = get_ext_augs(smi)
    augs = list(set(augs))
    
    augset = []
    if augs:
        for aug_smi in augs:
            tup = [anc_idx, src_idx, aug_iter, aug_smi]
            if tup not in augset:
                augset.append(tup)
    return augset

In [4]:
import numpy as np
from tqdm.notebook import tqdm

from extended_augs_utils import get_ext_augs
from joblib import Parallel, delayed

iters = 10
max_augs = 10

for aug_iter in tqdm(range(iters+1), total=(iters+1)):
    
    curr_augset = []
    
    if aug_iter==0:
        for row in df_samp.itertuples():
            idx = row.Index
            anc_idx = idx
            src_idx = idx
            smi = row.smiles
            tup = [idx, anc_idx, src_idx, aug_iter, smi]
            curr_augset.append(tup)
            df_augset = pd.DataFrame(curr_augset, columns=['idx','anc_idx','src_idx', 'aug_iter', 'smiles'])
        
    else:
        df_src = df_augset[df_augset.aug_iter==(aug_iter-1)]
        dict_src = df_src.T.to_dict('list')
            
        parallelizer = Parallel(n_jobs=-1, backend='multiprocessing' )
        tasks = (delayed(get_augset_from_src)(k,v,aug_iter) for k,v in dict_src.items())
        curr_augset = parallelizer(tasks)
        curr_augset = [x for l in curr_augset for x in l]
        
        
        print(len(df_src), len(curr_augset))
        
        
        curr_augset = pd.DataFrame(curr_augset, columns=['anc_idx', 'src_idx', 'aug_iter', 'smiles'])
        
        
        id_idc = [int(i) for i in range(len(df_src), len(curr_augset)+len(df_src))]
        print(len(id_idc))
#         print(id_idc)
        curr_augset['idx'] = id_idc
        curr_augset = curr_augset[['idx','anc_idx','src_idx', 'aug_iter', 'smiles']]
        display(curr_augset)
#         break
        
#         curr_augset = pd.DataFrame(curr_augset, columns=['anc_idx', 'src_idx', 'aug_iter', 'smiles'])
        curr_augset.drop_duplicates(keep='first', inplace=True)
#         display(curr_augset)
        
        trunc_idc = []
        for i in range(len(df_samp)):
            df_anc = curr_augset[curr_augset.anc_idx==i]
            df_anc = df_anc.sample(n=10, replace=True)
            idc = df_anc.index.values
            trunc_idc.extend(idc)
        trunc_augset = curr_augset.iloc[trunc_idc]
        
        df_augset = pd.concat([df_augset, trunc_augset], axis=0)
        
        display(df_augset)

  0%|          | 0/11 [00:00<?, ?it/s]

1000 9153
9153


,idx,anc_idx,src_idx,aug_iter,smiles
0,1000,0,0,1,CCCN(CC(O)CN)N=O
1,1001,0,0,1,CCCN(CC(C)(C)O)N=O
2,1002,0,0,1,CCCN(CC(C)OC)N=O
3,1003,0,0,1,CCCN(N=O)C(C)C(C)O
4,1004,0,0,1,CCC(C)N(CC(C)O)N=O
...,...,...,...,...,...
9148,10148,999,999,1,Cc1c([N+](=O)[O-])ccc(N2C(=O)c3cccc4cccc(c34)C...
9149,10149,999,999,1,Cc1cc([N+](=O)[O-])cc([N+](=O)[O-])c1N1C(=O)c2...
9150,10150,999,999,1,O=C1c2cccc3cc(S)cc(c23)C(=O)N1c1ccc([N+](=O)[O...
9151,10151,999,999,1,O=C1c2cccc3ccc(O)c(c23)C(=O)N1c1ccc([N+](=O)[O...


,idx,anc_idx,src_idx,aug_iter,smiles
0,0,0,0,0,CCCN(CC(C)O)N=O
1,1,1,1,0,Cn1cncc1C(=O)NN
2,2,2,2,0,S=c1ssc2c1CCCCC2
3,3,3,3,0,Cc1csc2nc(N)nn12
4,4,4,4,0,CCCOC(=O)c1ccccc1
...,...,...,...,...,...
9148,10148,999,999,1,Cc1c([N+](=O)[O-])ccc(N2C(=O)c3cccc4cccc(c34)C...
9145,10145,999,999,1,Cc1ccc2cccc3c2c1C(=O)N(c1ccc([N+](=O)[O-])cc1[...
9148,10148,999,999,1,Cc1c([N+](=O)[O-])ccc(N2C(=O)c3cccc4cccc(c34)C...
9147,10147,999,999,1,Cc1ccc2c3c(cccc13)C(=O)N(c1ccc([N+](=O)[O-])cc...


10000 58909
58909


...

,idx,anc_idx,src_idx,aug_iter,smiles
0,10000,0,10031,9,CCC(C(O)CCOC)N(CCC(C)(O)ON)N=O
1,10001,0,10031,9,CCC(C)(C(O)CCOC)N(CCC(C)(O)O)N=O
2,10002,0,10031,9,COCCC(O)C(C(C)C)N(CCC(C)(O)O)N=O
3,10003,0,10031,9,CCC(C(O)CCOC)N(CC(C)C(C)(O)O)N=O
4,10004,0,10031,9,CCC(C(CCOC)OC)N(CCC(C)(O)O)N=O
...,...,...,...,...,...
94097,104097,999,103558,9,CCc1c(CO)c2c3c(cc(SC)cc3c1C)C(=O)N(c1ccc([N+](...
94098,104098,999,103558,9,CSc1cc2c3c(c(CO)c(C)c(C)c3c1)C(=O)N(c1cc(C)c([...
94099,104099,999,103558,9,CSc1cc2c3c(c(CO)c(C)c(C)c3c1N)C(=O)N(c1ccc([N+...
94100,104100,999,103558,9,CSc1cc2c3c(c(C(C)O)c(C)c(C)c3c1)C(=O)N(c1ccc([...


,idx,anc_idx,src_idx,aug_iter,smiles
0,0,0,0,0,CCCN(CC(C)O)N=O
1,1,1,1,0,Cn1cncc1C(=O)NN
2,2,2,2,0,S=c1ssc2c1CCCCC2
3,3,3,3,0,Cc1csc2nc(N)nn12
4,4,4,4,0,CCCOC(=O)c1ccccc1
...,...,...,...,...,...
94070,104070,999,103585,9,CCc1cc([N+](=O)[O-])c(C)c([N+](=O)[O-])c1N1C(=...
94085,104085,999,103584,9,CCC(O)c1cc([N+](=O)[O-])c(C)c([N+](=O)[O-])c1N...
94014,104014,999,103622,9,CCCc1cc([N+](=O)[O-])c(C)c([N+](=O)[O-])c1N1C(...
94050,104050,999,103618,9,CCc1cc([N+](=O)[O-])c(C)c([N+](=O)[O-])c1N1C(=...


10000 93830
93830


,idx,anc_idx,src_idx,aug_iter,smiles
0,10000,0,10004,10,CCC(C(CCOC)OC)N(CCC(O)(O)CC)N=O
1,10001,0,10004,10,CCC(C(CCOC)OC)N(CCC(C)(O)OC)N=O
2,10002,0,10004,10,CCCC(C(CCOC)OC)N(CCC(C)(O)O)N=O
3,10003,0,10004,10,CCC(C(CCOC)OC)N(CC(C)C(C)(O)O)N=O
4,10004,0,10004,10,CCC(C)(C(CCOC)OC)N(CCC(C)(O)O)N=O
...,...,...,...,...,...
93825,103825,999,104071,10,CCc1cc([N+](=O)[O-])c(C)c([N+](=O)[O-])c1N1C(=...
93826,103826,999,104071,10,CCc1cc([N+](=O)[O-])c(C)c([N+](=O)[O-])c1N1C(=...
93827,103827,999,104071,10,CCc1cc([N+](=O)[O-])c(C)c([N+](=O)[O-])c1N1C(=...
93828,103828,999,104071,10,CCc1cc([N+](=O)[O-])c(C)c([N+](=O)[O-])c1N1C(=...


,idx,anc_idx,src_idx,aug_iter,smiles
0,0,0,0,0,CCCN(CC(C)O)N=O
1,1,1,1,0,Cn1cncc1C(=O)NN
2,2,2,2,0,S=c1ssc2c1CCCCC2
3,3,3,3,0,Cc1csc2nc(N)nn12
4,4,4,4,0,CCCOC(=O)c1ccccc1
...,...,...,...,...,...
93738,103738,999,104043,10,CCc1cc([N+](=O)[O-])c(C)c([N+](=O)[O-])c1N1C(=...
93803,103803,999,104014,10,CCCc1cc([N+](=O)[O-])c(C)c([N+](=O)[O-])c1N1C(...
93809,103809,999,104014,10,CCCCc1cc([N+](=O)[O-])c(C)c([N+](=O)[O-])c1N1C...
93749,103749,999,104056,10,CCc1cc([N+](=O)[O-])c(C)c([N+](=O)[O-])c1N1C(=...


In [5]:
df_augset

,idx,anc_idx,src_idx,aug_iter,smiles
0,0,0,0,0,CCCN(CC(C)O)N=O
1,1,1,1,0,Cn1cncc1C(=O)NN
2,2,2,2,0,S=c1ssc2c1CCCCC2
3,3,3,3,0,Cc1csc2nc(N)nn12
4,4,4,4,0,CCCOC(=O)c1ccccc1
...,...,...,...,...,...
93738,103738,999,104043,10,CCc1cc([N+](=O)[O-])c(C)c([N+](=O)[O-])c1N1C(=...
93803,103803,999,104014,10,CCCc1cc([N+](=O)[O-])c(C)c([N+](=O)[O-])c1N1C(...
93809,103809,999,104014,10,CCCCc1cc([N+](=O)[O-])c(C)c([N+](=O)[O-])c1N1C...
93749,103749,999,104056,10,CCc1cc([N+](=O)[O-])c(C)c([N+](=O)[O-])c1N1C(=...


In [6]:
df_augset_nodups = df_augset.drop_duplicates(keep='first')
df_augset_nodups

,idx,anc_idx,src_idx,aug_iter,smiles
0,0,0,0,0,CCCN(CC(C)O)N=O
1,1,1,1,0,Cn1cncc1C(=O)NN
2,2,2,2,0,S=c1ssc2c1CCCCC2
3,3,3,3,0,Cc1csc2nc(N)nn12
4,4,4,4,0,CCCOC(=O)c1ccccc1
...,...,...,...,...,...
93794,103794,999,104085,10,CCc1c([N+](=O)[O-])cc(C(O)CC)c(N2C(=O)c3cccc4c...
93738,103738,999,104043,10,CCc1cc([N+](=O)[O-])c(C)c([N+](=O)[O-])c1N1C(=...
93803,103803,999,104014,10,CCCc1cc([N+](=O)[O-])c(C)c([N+](=O)[O-])c1N1C(...
93809,103809,999,104014,10,CCCCc1cc([N+](=O)[O-])c(C)c([N+](=O)[O-])c1N1C...


In [7]:
df_augset_nodups.to_csv('20220603_extended_augset_1000anc_10iters.csv',index=False)